# SPÉ 1 — Notebook de démarrage : métastases dans des images histopathologiques

**MBA 2 Data & BI — Epitech — 2026-2027**

Ce notebook vous donne une base qui fonctionne de bout en bout, à reprendre et à améliorer :

1. charger le sous-échantillon de Camelyon17-WILDS (39 000 patchs) et le **découper par lame** ;
2. explorer les données par hôpital, par lame et par classe ;
3. entraîner une **référence simple** (histogrammes de couleur + régression logistique) ;
4. entraîner un **CNN pré-entraîné** (ResNet18) par transfert, avec **trois traitements de la couleur** comparés ;
5. évaluer **hôpital par hôpital**, choisir le seuil selon le coût des faux négatifs ;
6. regarder ce que voit le modèle avec **Grad-CAM** (patchs bien et mal classés) ;
7. remonter du patch à la lame (**étendue des métastases, classe de la lame**) puis au **stade pN** ;
8. produire la **synthèse chiffrée du follow-up** (`resultats_followup.md` + figures) ;
9. lancer le **test final** sur l'hôpital 2, une seule fois.

**Avant de commencer** (Colab) : menu *Exécution → Modifier le type d'exécution → GPU (T4)*.

Ce notebook est un point de départ, pas une solution : chaque choix (taille d'image, nombre d'époques, seuil, agrégation…) est à questionner et à justifier dans votre rapport.

## 0. Paramètres

In [ ]:
# Où trouver le sous-échantillon ?
#  "drive" : le fichier camelyon17_sous_echantillon.parquet a été copié dans votre Google Drive (le plus rapide)
#  "hf"    : reconstruction automatique depuis Hugging Face (environ 10 Go téléchargés, 10 à 20 minutes sur Colab)
SOURCE = "hf"
DRIVE_PATH = "/content/drive/MyDrive/SPE1/camelyon17_sous_echantillon.parquet"

# Fichiers du pack élèves, à déposer à côté du notebook (ou dans Colab : panneau Fichiers → Importer)
IDS_CSV = "data/ids_sous_echantillon.csv"     # liste des patchs du sous-échantillon
META_ZIP = "data/metadata_complete.zip"       # métadonnées des 455 954 patchs (sans images)
STAGES_CSV = "data/stages.csv"                # classe de chaque lame et stade pN des 100 patientes

# Colab n'ouvre que le notebook : si data/ est absent, on clone le dépôt et on se place dedans
import os
if not os.path.exists("data"):
    if not os.path.exists("model-cancer-sein"):
        os.system("git clone -q https://github.com/Cloud-io09/model-cancer-sein.git")
    os.chdir("model-cancer-sein")

QUICK = False        # True = petit essai rapide (quelques centaines de patchs) pour vérifier que tout marche
EPOCHS_HEAD = 2      # époques avec seule la dernière couche entraînée
EPOCHS_FT = 2        # époques d'ajustement fin (dernier bloc + dernière couche)
SEED = 42
NUM_WORKERS = 2      # processus qui décodent les images en parallèle (Colab : 2 cœurs)

# Choix méthodologiques, à justifier dans le rapport
TAILLE = 96                               # entrée du CNN : 96 (taille native) ou 224 (agrandi, environ 5 fois plus lent)
VARIANTES = ["base", "couleur", "gris"]   # traitements de la couleur comparés en section 5
CIBLE_RAPPEL = 0.95                       # rappel minimal visé sur val_ood pour fixer le seuil des patchs
SEUIL_LAME = 0.5                          # seuil des patchs pour dessiner les zones tumorales d'une lame (section 8)
LAMES_H1 = True                           # section 8 : prédire tous les patchs des 10 lames de l'hôpital 1 (~1,5 Go téléchargés)

# Géométrie des patchs (WILDS, annexe E.2) : niveau de résolution ÷4 d'un scan à 0,23–0,25 µm/pixel,
# patchs de 96 × 96 px posés tous les 32 px ; label = tumeur dans le carré central de 32 × 32 px
UM_PAR_PX = 0.24 * 4
PAS = 32

## 1. Installation et imports

In [ ]:
import os, io, json, time, random, zipfile, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image
from scipy import ndimage
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist
import torch, torch.nn as nn, torchvision
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve, recall_score, precision_score, accuracy_score, confusion_matrix

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Calcul sur :", DEVICE)
os.makedirs("figures", exist_ok=True)
def sauver(nom):
    """Enregistre la figure courante dans figures/, pour les diapos et le rapport."""
    plt.savefig(f"figures/{nom}.png", dpi=150, bbox_inches="tight")

HOPITAL = {0: "H0 (entr.)", 1: "H1 (valid. hors distrib.)", 2: "H2 (test, jamais vu)", 3: "H3 (entr.)", 4: "H4 (entr.)"}

## 2. Charger le sous-échantillon

Le sous-échantillon contient 39 000 patchs de 96 × 96 pixels, tirés au hasard et **équilibrés par hôpital et par classe** :

| Jeu | Hôpitaux | Patchs | Rôle |
|---|---|---|---|
| `train` | 0, 3, 4 | 24 000 | entraînement |
| `id_val` | 0, 3, 4 | 3 000 | validation « même distribution » (mêmes hôpitaux que l'entraînement) |
| `val_ood` | 1 | 4 000 | validation « hors distribution » : réglages et choix du modèle |
| `test` | 2 | 8 000 | **test final, à n'utiliser qu'une fois** |

Attention : `id_val` vient des mêmes lames que `train`. C'est le découpage officiel de WILDS, mais il surestime la performance. Juste après le chargement, on refait donc ce découpage **par patiente**.

In [ ]:
API_HF = "https://huggingface.co/api/datasets/wltjr1007/Camelyon17-WILDS/parquet/default"

def reconstruire_depuis_hf(ids_csv, sortie="camelyon17_sous_echantillon.parquet"):
    """Télécharge les fichiers Parquet de Hugging Face un par un, garde les patchs du sous-échantillon, efface le reste."""
    import pyarrow.parquet as pq, pyarrow as pa
    ids = pd.read_csv(ids_csv)
    voulu = dict(zip(ids.image_id, ids.split))
    shards = json.load(urllib.request.urlopen(API_HF))
    morceaux = []
    for split, urls in shards.items():
        for i, url in enumerate(urls):
            t0 = time.time(); f = f"tmp_{split}_{i}.parquet"
            urllib.request.urlretrieve(url, f)
            t = pq.read_table(f)
            ids_shard = t.column("image_id").to_numpy()
            garder = np.flatnonzero(np.isin(ids_shard, list(voulu)))
            if len(garder):
                s = t.take(pa.array(garder)).to_pandas()
                s["image"] = s["image"].apply(lambda d: d["bytes"])
                morceaux.append(s)
            os.remove(f)
            print(f"{split} {i + 1}/{len(urls)} : {len(garder)} patchs gardés ({time.time() - t0:.0f} s)")
    df = pd.concat(morceaux, ignore_index=True)
    df["split"] = df.image_id.map(voulu)
    df.to_parquet(sortie, index=False)
    return sortie

if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    chemin = DRIVE_PATH
else:
    chemin = "camelyon17_sous_echantillon.parquet"
    if not os.path.exists(chemin):
        chemin = reconstruire_depuis_hf(IDS_CSV, chemin)

df = pd.read_parquet(chemin)
if QUICK:
    df = df.sample(frac=1, random_state=SEED).groupby(["split", "center", "label"]).head(40)
df = df.reset_index(drop=True)
print(len(df), "patchs")
df.groupby(["split", "center", "label"]).size().unstack()

### Découpage par patiente

Des patchs d'une même lame (ou d'une même patiente) se ressemblent beaucoup : les répartir au hasard entre entraînement et validation gonfle les scores. On fusionne donc `train` et `id_val`, puis on met de côté des **patientes entières** de chaque hôpital d'entraînement : c'est `val_lames`. Les patientes sont tirées jusqu'à couvrir au moins 10 % (et au plus 40 %) des patchs de chaque classe de l'hôpital, parce que la part de tumeur varie énormément d'une lame à l'autre.

| Ensemble | Rôle |
|---|---|
| `entr` | entraînement |
| `val_lames` | hôpitaux connus, lames jamais vues : performance « même distribution » sans fuite |
| `val_ood` | hôpital 1 jamais vu : choix du modèle et des seuils |
| `test` | hôpital 2 : test final, une seule fois (section 10) |

In [ ]:
def choisir_patientes(d, rng, mini=.1, maxi=.4):
    """Tire des patientes au hasard jusqu'à couvrir au moins `mini` des patchs de chaque classe, sans dépasser `maxi`."""
    total = d.groupby("label").size()
    for _ in range(1000):
        choix = []
        for p in rng.permutation(d.patient.unique()):
            choix.append(p)
            part = d[d.patient.isin(choix)].groupby("label").size().reindex(total.index, fill_value=0) / total
            if (part >= mini).all():
                break
        if (part <= maxi).all():
            return choix
    print("Attention : aucun tirage ne respecte les bornes, le dernier est gardé")
    return choix

rng = np.random.default_rng(SEED)
df["ens"] = df.split.replace({"train": "entr", "id_val": "entr"})
for h in (0, 3, 4):
    d = df[(df.ens == "entr") & (df.center == h)]
    df.loc[d.index[d.patient.isin(choisir_patientes(d, rng))], "ens"] = "val_lames"
assert not set(df[df.ens == "entr"].slide) & set(df[df.ens == "val_lames"].slide), "une lame est dans deux ensembles"
ENS = ["entr", "val_lames", "val_ood"]   # le test (hôpital 2) reste à part jusqu'à la section 10
df.groupby(["ens", "center"]).agg(patchs=("label", "size"), lames=("slide", "nunique"),
                                  patientes=("patient", "nunique"), part_tumorale=("label", "mean")).round(2)

## 3. Exploration

In [ ]:
def ouvrir(b):
    return Image.open(io.BytesIO(b)).convert("RGB")

fig, axes = plt.subplots(5, 8, figsize=(14, 9))
for r, h in enumerate(range(5)):
    for c in range(8):
        lab = c // 4
        s = df[(df.center == h) & (df.label == lab)].sample(1, random_state=c)
        axes[r, c].imshow(ouvrir(s.image.iloc[0])); axes[r, c].axis("off")
        if c in (0, 4):
            axes[r, c].set_title(f"{HOPITAL[h]}\n{'tumoral' if lab else 'normal'}", fontsize=8)
plt.suptitle("4 patchs normaux puis 4 patchs tumoraux, pour chaque hôpital"); plt.tight_layout(); sauver("01_patchs_par_hopital"); plt.show()

In [ ]:
# La couleur change-t-elle d'un hôpital à l'autre ? Moyenne des canaux RGB par patch
def moyenne_rgb(b):
    return np.asarray(ouvrir(b), dtype=np.float32).reshape(-1, 3).mean(0)

ech = df.sample(frac=1, random_state=SEED).groupby("center").head(300)
rgb = np.stack(ech.image.map(moyenne_rgb).values)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for k, canal in enumerate(["Rouge", "Vert", "Bleu"]):
    for h in range(5):
        axes[k].hist(rgb[ech.center.values == h, k], bins=30, alpha=.45, label=HOPITAL[h], density=True)
    axes[k].set_title(canal)
axes[0].legend(fontsize=7); plt.tight_layout(); sauver("02_couleurs_par_hopital"); plt.show()

### Répartition complète par hôpital, par lame et par classe

Les métadonnées décrivent les 455 954 patchs de WILDS (sans les images). Le nombre de patchs et la part de tumeur varient énormément d'une lame à l'autre : une lame peut peser plus que tout un hôpital.

In [ ]:
with zipfile.ZipFile(META_ZIP) as z:
    meta = pd.read_csv(z.open("metadata_complete.csv"))
stages = pd.read_csv(STAGES_CSV)
lames = meta.groupby("slide").agg(patient=("patient", "first"), node=("node", "first"), center=("center", "first"),
                                   n_patchs=("label", "size"), part_tumorale=("label", "mean")).reset_index()
lames["nom"] = [f"patient_{p:03d}_node_{n}.tif" for p, n in zip(lames.patient, lames.node)]
lames = lames.merge(stages.rename(columns={"patient": "nom", "stage": "classe_officielle"})[["nom", "classe_officielle"]], on="nom")

display(lames.groupby("center").agg(lames=("slide", "size"), patientes=("patient", "nunique"), patchs=("n_patchs", "sum"),
                                    patchs_min=("n_patchs", "min"), patchs_max=("n_patchs", "max"),
                                    part_tum_min=("part_tumorale", "min"), part_tum_max=("part_tumorale", "max")).round(3))
display(pd.crosstab(lames.center.map(HOPITAL), lames.classe_officielle))

o = lames.sort_values(["center", "n_patchs"])
couleurs = plt.cm.tab10(o.center.values)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].bar(range(len(o)), o.n_patchs, color=couleurs); axes[0].set_yscale("log"); axes[0].set_title("Patchs par lame (échelle log)")
axes[1].bar(range(len(o)), o.part_tumorale, color=couleurs); axes[1].set_title("Part de patchs tumoraux par lame")
for a in axes:
    a.set_xlabel("lames, groupées par hôpital"); a.set_xticks([])
axes[0].legend(handles=[Patch(color=plt.cm.tab10(h), label=HOPITAL[h]) for h in range(5)], fontsize=7)
plt.tight_layout(); sauver("03_patchs_par_lame"); plt.show()

### La couleur suffit-elle à reconnaître l'hôpital ?

Si un simple histogramme de couleur reconnaît l'hôpital d'un patch, un CNN peut lui aussi s'appuyer sur la couleur plutôt que sur les cellules, et échouer dans un hôpital qu'il n'a jamais vu. La validation croisée est groupée par lame : on reconnaît l'hôpital sur des lames jamais vues.

In [ ]:
def hist_couleur(b, bins=16):
    a = np.asarray(ouvrir(b))
    return np.concatenate([np.histogram(a[..., k], bins=bins, range=(0, 255), density=True)[0] for k in range(3)])

Xh = np.stack(ech.image.map(hist_couleur).values)
acc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), Xh, ech.center,
                      groups=ech.slide, cv=StratifiedGroupKFold(5, shuffle=True, random_state=SEED))
print(f"Hôpital reconnu à partir de la seule couleur : {acc.mean():.1%} de bonnes réponses (hasard : 20 %)")

### Trois constats d'exploration (follow-up)

Calculés ici pour être repris tels quels dans la présentation et le rapport.

In [ ]:
nb = lambda n: f"{n:,}".replace(",", " ")
plus_grosse = lames.loc[lames.n_patchs.idxmax()]
CONSTATS = [
    f"La couleur trahit l'hôpital : un histogramme de couleur reconnaît l'hôpital de lames jamais vues dans {acc.mean():.0%} des cas "
    f"(hasard : 20 %). Un modèle peut donc apprendre l'hôpital au lieu des cellules.",
    f"Les lames sont très inégales : de {nb(lames.n_patchs.min())} à {nb(lames.n_patchs.max())} patchs par lame. La plus grosse "
    f"(hôpital {plus_grosse.center}) en compte {nb(plus_grosse.n_patchs)}, l'hôpital 1 entier {nb(lames[lames.center == 1].n_patchs.sum())}. "
    f"D'où un découpage par patiente et des résultats par hôpital.",
    f"La part de tumeur va de {lames.part_tumorale.min():.1%} à {lames.part_tumorale.max():.0%} selon la lame, et WILDS est équilibré "
    f"à 50 % par construction : en clinique, la précision sera bien plus basse qu'ici.",
]
for k, c in enumerate(CONSTATS, 1):
    print(f"{k}. {c}\n")

## 4. Référence simple : histogrammes de couleur + régression logistique

Un modèle complexe ne vaut que s'il bat une référence simple. Ici : 16 classes par canal RGB, soit 48 variables par patch. Tous les résultats de validation sont rangés dans `RESULTATS` pour le tableau comparatif de la section 5.

In [ ]:
def evaluer(d, p, seuil=0.5):
    """Métriques par hôpital. d : patchs (colonnes center, label) ; p : probabilité prédite d'être tumoral."""
    lignes = []
    for h, g in d.assign(p=p).groupby("center"):
        pred = g.p >= seuil
        lignes.append({"hôpital": HOPITAL[h], "n": len(g),
                       "ROC-AUC": roc_auc_score(g.label, g.p) if g.label.nunique() == 2 else np.nan,
                       "rappel": recall_score(g.label, pred, zero_division=0),
                       "précision": precision_score(g.label, pred, zero_division=0),
                       "accuracy": accuracy_score(g.label, pred)})
    return pd.DataFrame(lignes).round(3)

RESULTATS = []   # une ligne par modèle × ensemble × hôpital
def noter(nom, ens, p):
    RESULTATS.extend(evaluer(df[df.ens == ens], p).assign(modèle=nom, ensemble=ens).to_dict("records"))

X = {s: np.stack(df[df.ens == s].image.map(hist_couleur).values) for s in ENS}
y = {s: df[df.ens == s].label.values for s in ENS}
ref = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X["entr"], y["entr"])
for s in ["val_lames", "val_ood"]:
    p = ref.predict_proba(X[s])[:, 1]
    noter("référence", s, p)
    print(f"{s:9s} ROC-AUC = {roc_auc_score(y[s], p):.3f}   rappel = {recall_score(y[s], p > .5):.3f}")

## 5. CNN pré-entraîné par transfert (ResNet18)

Transfert en deux temps : d'abord seule la dernière couche apprend, puis le dernier bloc (`layer4`) avec elle. Les patchs font 96 × 96 ; les agrandir à 224 × 224 (`TAILLE`) est une piste à tester et à justifier.

Pour réduire l'écart entre hôpitaux, on compare trois traitements de la couleur, entraînés de la même façon :

| Variante | Idée |
|---|---|
| `base` | retournements horizontaux et verticaux seulement |
| `couleur` | + variations aléatoires de luminosité, contraste, saturation et teinte : le modèle voit des colorations variées |
| `gris` | images en niveaux de gris, à l'entraînement comme à l'évaluation : la couleur de l'hôpital disparaît |

Le jeu est équilibré (50 % de patchs tumoraux dans chaque hôpital), donc la perte n'est pas pondérée. En clinique, la tumeur est rare : voir la section 6.

In [ ]:
MOY, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # normalisation ImageNet
T = torchvision.transforms

def transfo(variante, entr):
    t = [T.Resize(TAILLE)] if TAILLE != 96 else []
    if entr:
        t += [T.RandomHorizontalFlip(), T.RandomVerticalFlip()]
        if variante == "couleur":
            t += [T.ColorJitter(brightness=.25, contrast=.25, saturation=.25, hue=.05)]
    if variante == "gris":
        t += [T.Grayscale(3)]
    return T.Compose(t + [T.ToTensor(), T.Normalize(MOY, STD)])

class Patchs(Dataset):
    def __init__(self, d, tf):
        self.b, self.y, self.tf = d.image.values, d.label.values, tf
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.tf(ouvrir(self.b[i])), int(self.y[i])

def chargeur(ens, tf, shuffle=False):
    return DataLoader(Patchs(df[df.ens == ens], tf), batch_size=128, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

@torch.no_grad()
def probas(m, dl):
    m.eval(); out = []
    for x, _ in dl:
        out.append(torch.softmax(m(x.to(DEVICE)), 1)[:, 1].cpu())
    return torch.cat(out).numpy()

def entrainer(m, params, epochs, lr, dl_entr, dl_val):
    opt = torch.optim.Adam(params, lr=lr); crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train(); t0 = time.time(); perte = 0
        for x, yy in dl_entr:
            x, yy = x.to(DEVICE), yy.to(DEVICE)
            opt.zero_grad(); l = crit(m(x), yy); l.backward(); opt.step(); perte += l.item() * len(yy)
        auc = roc_auc_score(dl_val.dataset.y, probas(m, dl_val))
        print(f"époque {ep + 1}/{epochs}  perte {perte / len(dl_entr.dataset):.3f}  ROC-AUC val_ood {auc:.3f}  ({time.time() - t0:.0f} s)")

def nouveau_resnet():
    m = torchvision.models.resnet18(weights="IMAGENET1K_V1")
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

def entrainer_variante(v):
    torch.manual_seed(SEED)
    m = nouveau_resnet()
    dl_entr, dl_val = chargeur("entr", transfo(v, True), shuffle=True), chargeur("val_ood", transfo(v, False))
    # Étape 1 : seule la dernière couche apprend
    for p in m.parameters():
        p.requires_grad = False
    for p in m.fc.parameters():
        p.requires_grad = True
    entrainer(m, m.fc.parameters(), EPOCHS_HEAD, 1e-3, dl_entr, dl_val)
    # Étape 2 : ajustement fin du dernier bloc
    for p in m.layer4.parameters():
        p.requires_grad = True
    entrainer(m, list(m.layer4.parameters()) + list(m.fc.parameters()), EPOCHS_FT, 1e-4, dl_entr, dl_val)
    return m

In [ ]:
modeles, P = {}, {}   # P[variante][ensemble] : probabilités prédites, dans l'ordre de df[df.ens == ensemble]
for v in VARIANTES:
    print(f"--- variante {v}")
    modeles[v] = entrainer_variante(v)
    torch.save(modeles[v].state_dict(), f"resnet18_{v}.pt")
    P[v] = {s: probas(modeles[v], chargeur(s, transfo(v, False))) for s in ["val_lames", "val_ood"]}
    for s in P[v]:
        noter(f"resnet18_{v}", s, P[v][s])

In [ ]:
# ROC-AUC par hôpital : val_lames = hôpitaux connus mais lames jamais vues, val_ood = hôpital 1 jamais vu
display(pd.DataFrame(RESULTATS).pivot_table(index="modèle", columns=["ensemble", "hôpital"], values="ROC-AUC").round(3))

# Le modèle est choisi sur val_ood, jamais sur le test
auc_ood = {v: roc_auc_score(df[df.ens == "val_ood"].label, P[v]["val_ood"]) for v in VARIANTES}
CHOIX = max(auc_ood, key=auc_ood.get)
modele, tf_eval = modeles[CHOIX], transfo(CHOIX, False)
print("ROC-AUC val_ood :", {v: round(a, 3) for v, a in auc_ood.items()}, "→ modèle retenu :", CHOIX)

## 6. Évaluation du modèle retenu

Le seuil de 0,5 n'est qu'un point de départ. **Un faux négatif est une métastase manquée** : on choisit le seuil sur `val_ood` pour atteindre le rappel visé (`CIBLE_RAPPEL`), jamais sur le test.

In [ ]:
d_v, p_v = df[df.ens == "val_ood"], P[CHOIX]["val_ood"]
seuils = np.linspace(0.01, 0.99, 99)
rappels = np.array([recall_score(d_v.label, p_v >= s) for s in seuils])
SEUIL = seuils[rappels >= CIBLE_RAPPEL].max() if (rappels >= CIBLE_RAPPEL).any() else 0.5
print(f"Seuil retenu sur val_ood pour un rappel ≥ {CIBLE_RAPPEL:.0%} : {SEUIL:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for k, s in enumerate(["val_lames", "val_ood"]):
    d, p = df[df.ens == s], P[CHOIX][s]
    print(s); display(evaluer(d, p, SEUIL))
    cm = confusion_matrix(d.label, p >= SEUIL, labels=[0, 1])
    axes[k].imshow(cm, cmap="Blues")
    for (i, j), n in np.ndenumerate(cm):
        axes[k].text(j, i, n, ha="center", va="center")
    axes[k].set_xticks([0, 1], ["normal", "tumoral"]); axes[k].set_yticks([0, 1], ["normal", "tumoral"])
    axes[k].set_xlabel("prédit"); axes[k].set_ylabel("vrai"); axes[k].set_title(f"{s}, seuil {SEUIL:.2f}")
    for h, g in d.assign(p=p).groupby("center"):
        if g.label.nunique() == 2:
            fpr, tpr, _ = roc_curve(g.label, g.p)
            axes[2].plot(fpr, tpr, label=HOPITAL[h])
axes[2].plot([0, 1], [0, 1], "k--", lw=.8); axes[2].legend(fontsize=7)
axes[2].set_xlabel("taux de faux positifs"); axes[2].set_ylabel("rappel"); axes[2].set_title("Courbes ROC par hôpital")
plt.tight_layout(); sauver("04_confusion_et_roc"); plt.show()

### Précision attendue en conditions réelles

Le sous-échantillon contient 50 % de patchs tumoraux ; une vraie lame, beaucoup moins. À rappel et spécificité égaux, la précision chute quand la tumeur devient rare.

In [ ]:
vn, fp, fn, vp = confusion_matrix(d_v.label, p_v >= SEUIL, labels=[0, 1]).ravel()
sens, spec = vp / (vp + fn), vn / (vn + fp)
prev = np.array([.5, .2, .05, .01])
ppv = sens * prev / (sens * prev + (1 - spec) * (1 - prev))
print(f"Au seuil {SEUIL:.2f} sur val_ood : rappel {sens:.3f}, spécificité {spec:.3f}")
pd.DataFrame({"part de patchs tumoraux": prev, "précision attendue": ppv.round(3),
              "faux positifs pour 1 vrai positif": ((1 - ppv) / ppv).round(1)})

## 7. Que regarde le modèle ? Grad-CAM

Grad-CAM colore les zones du patch qui ont le plus pesé dans la décision « tumoral ». On regarde des vrais positifs (VP), faux négatifs (FN), faux positifs (FP) et vrais négatifs (VN), sur un hôpital connu et sur l'hôpital 1 : le modèle regarde-t-il les noyaux des cellules, ou la teinte générale ?

In [ ]:
def gradcam(m, x):
    acts = {}
    h = m.layer4.register_forward_hook(lambda mod, i, o: acts.__setitem__("a", o))
    m.eval()
    out = m(x.unsqueeze(0).to(DEVICE)); h.remove()
    g = torch.autograd.grad(out[0, 1], acts["a"])[0]
    w = g.mean((2, 3), keepdim=True)
    cam = torch.relu((w * acts["a"]).sum(1)).squeeze().detach().cpu().numpy()
    cam = np.array(Image.fromarray(cam / (cam.max() + 1e-8)).resize((96, 96), Image.BILINEAR))
    return cam, torch.softmax(out, 1)[0, 1].item()

def exemples(ens, n=2):
    """n patchs de chaque catégorie (VP, FN, FP, VN) au seuil retenu."""
    d = df[df.ens == ens].assign(p=P[CHOIX][ens]); pred = d.p >= SEUIL
    cats = {"VP": (d.label == 1) & pred, "FN": (d.label == 1) & ~pred, "FP": (d.label == 0) & pred, "VN": (d.label == 0) & ~pred}
    return [(c, r) for c, masque in cats.items() for _, r in d[masque].sample(min(n, masque.sum()), random_state=SEED).iterrows()]

for p in modele.parameters():
    p.requires_grad = True
for ens in ["val_lames", "val_ood"]:
    ex = exemples(ens)
    fig, axes = plt.subplots(2, len(ex), figsize=(2 * len(ex), 4.5), squeeze=False)
    for k, (cat, r) in enumerate(ex):
        img = ouvrir(r.image); cam, p = gradcam(modele, tf_eval(img))
        axes[0, k].imshow(img); axes[0, k].set_title(f"{cat} {HOPITAL[r.center][:2]} p={p:.2f}", fontsize=8)
        axes[1, k].imshow(img); axes[1, k].imshow(cam, cmap="jet", alpha=.45)
        axes[0, k].axis("off"); axes[1, k].axis("off")
    plt.suptitle(f"Grad-CAM, {ens}"); plt.tight_layout(); sauver(f"05_gradcam_{ens}"); plt.show()

## 8. Du patch à la lame, puis à la patiente

**Étendue d'une métastase.** Les patchs sont posés tous les 32 px sur une grille, et chaque patch porte le label de son carré central de 32 × 32 px, soit environ 31 µm de côté (`PAS × UM_PAR_PX`). WILDS a gardé **tous** les patchs tumoraux de chaque lame (sauf une), et seulement une partie des patchs normaux. Sur la grille, une zone tumorale est un groupe de cases tumorales voisines (8-connexité). Son diamètre est la plus grande distance entre deux de ses cases, plus une case, ce qui l'arrondit plutôt vers le haut, donc vers la classe la plus grave. La classe de la lame découle de la plus grande zone : ITC ≤ 0,2 mm < micro ≤ 2 mm < macro.

**Plafond de la méthode.** Avant tout modèle, on applique cette mesure aux labels vrais des 50 lames : l'écart restant avec la classe officielle vient de la grille elle-même, pas du CNN. La définition officielle des ITC compte aussi le nombre de cellules (moins de 200), ce que la grille ne voit pas.

Rappel : les 50 lames de WILDS sont toutes positives, et aucune patiente n'a ses 5 ganglions annotés.

In [ ]:
CELLULE_MM = PAS * UM_PAR_PX / 1000   # côté d'une case de la grille, environ 0,03 mm

def diametre(q):
    """Plus grande distance entre deux cases (en cases). L'enveloppe convexe évite de comparer tous les couples."""
    if len(q) > 50:
        try:
            q = q[ConvexHull(q).vertices]
        except Exception:   # cases alignées : pas d'enveloppe, on garde tous les points
            pass
    return pdist(q).max() if len(q) > 1 else 0

def etendue_mm(g, col, seuil):
    """Diamètre (mm) de la plus grande zone tumorale d'une lame ; une case est tumorale si g[col] >= seuil."""
    t = g[g[col] >= seuil]
    if t.empty:
        return 0.0
    pts = np.column_stack([t.x_coord.values // PAS, t.y_coord.values // PAS])
    pts -= pts.min(0)
    grille = np.zeros(pts.max(0) + 1, bool); grille[pts[:, 0], pts[:, 1]] = True
    zones, _ = ndimage.label(grille, structure=np.ones((3, 3)))
    ids = zones[pts[:, 0], pts[:, 1]]
    ordre = np.argsort(ids); pts, ids = pts[ordre], ids[ordre]
    meilleur = 0
    for q in np.split(pts, np.unique(ids, return_index=True)[1][1:]):
        if np.hypot(*(q.max(0) - q.min(0))) + 1 > meilleur:   # sinon la zone ne peut pas battre la meilleure
            meilleur = max(meilleur, diametre(q) + 1)
    return meilleur * CELLULE_MM

def classe_lame(mm):
    return "negative" if mm == 0 else "itc" if mm <= 0.2 else "micro" if mm <= 2 else "macro"

CLASSES = ["negative", "itc", "micro", "macro"]
def confusion_lames(vrai, pred):
    return pd.crosstab(pd.Categorical(vrai, CLASSES), pd.Categorical(pred, CLASSES),
                       rownames=["officielle"], colnames=["prédite"], dropna=False)

lames["mm_vrai"] = [etendue_mm(meta[meta.slide == s], "label", 1) for s in lames.slide]
lames["classe_grille"] = lames.mm_vrai.map(classe_lame)
ACCORD_GRILLE = (lames.classe_grille == lames.classe_officielle).mean()
print(f"Labels vrais → classe de la lame : {ACCORD_GRILLE:.0%} d'accord avec la classe officielle (50 lames)")
confusion_lames(lames.classe_officielle, lames.classe_grille)

### Cartes de chaleur des lames de l'hôpital 1

Le sous-échantillon ne contient qu'une petite partie des patchs de chaque lame. On télécharge donc **tous** les patchs des 10 lames de l'hôpital 1 (~1,5 Go), on les prédit fichier par fichier sans garder les images, et on met les probabilités en cache dans un CSV. Effacez ce CSV si vous réentraînez le modèle dans la même session.

Les lames des hôpitaux 0, 3 et 4 ont servi à l'entraînement : les prédire ne mesurerait rien. L'hôpital 2 est réservé au test final.

Limite : WILDS n'a gardé qu'une partie des patchs normaux, donc les faux positifs sur une vraie lame entière seraient plus nombreux que ceux observés ici.

In [ ]:
def predire_lames_hf(split_hf, centre, m, tf, sortie):
    """Prédit tous les patchs d'un hôpital, fichier Parquet par fichier, sans garder les images. Résultat mis en cache."""
    if os.path.exists(sortie):
        return pd.read_csv(sortie)
    import pyarrow.parquet as pq
    urls = json.load(urllib.request.urlopen(API_HF))[split_hf]
    morceaux = []
    for i, url in enumerate(urls):
        t0 = time.time(); f = f"tmp_{split_hf}_{i}.parquet"
        urllib.request.urlretrieve(url, f)
        s = pq.read_table(f, filters=[("center", "=", centre)]).to_pandas()
        os.remove(f)
        if len(s):
            s["image"] = s["image"].apply(lambda d: d["bytes"])
            s["p"] = probas(m, DataLoader(Patchs(s, tf), batch_size=256, num_workers=NUM_WORKERS))
            morceaux.append(s.drop(columns="image"))
        print(f"{split_hf} {i + 1}/{len(urls)} : {len(s)} patchs de l'hôpital {centre} prédits ({time.time() - t0:.0f} s)")
    d = pd.concat(morceaux, ignore_index=True)
    d.to_csv(sortie, index=False)
    return d

if LAMES_H1:
    toutes_h1 = predire_lames_hf("validation", 1, modele, tf_eval, f"predictions_{CHOIX}_H1.csv")
    print(len(toutes_h1), "patchs,", toutes_h1.slide.nunique(), "lames ; ROC-AUC sur tous les patchs :",
          round(roc_auc_score(toutes_h1.label, toutes_h1.p), 3))
    ids_h1 = sorted(toutes_h1.slide.unique())
    rangs = (len(ids_h1) + 1) // 2
    fig, axes = plt.subplots(rangs, 4, figsize=(16, 3.6 * rangs), squeeze=False)
    for a in axes.ravel():
        a.axis("off")
    for k, s in enumerate(ids_h1):
        g = toutes_h1[toutes_h1.slide == s]
        for j, (col, titre) in enumerate([("label", "vérité"), ("p", "probabilité prédite")]):
            a = axes[k // 2, 2 * (k % 2) + j]
            a.scatter(g.x_coord, g.y_coord, c=g[col], s=1, cmap="coolwarm", vmin=0, vmax=1)
            a.set_title(f"lame {s} : {titre}", fontsize=9); a.invert_yaxis(); a.set_aspect("equal")
    plt.tight_layout(); sauver("06_cartes_lames_H1"); plt.show()

In [ ]:
# Classe de chaque lame de l'hôpital 1 à partir des prédictions, comparée à la classe officielle.
# SEUIL_LAME se règle ici, sur l'hôpital 1 : ce résultat est donc optimiste, seul le test (section 10) est honnête.
if LAMES_H1:
    l1 = lames[lames.slide.isin(ids_h1)].copy()
    for s in sorted({0.5, 0.7, 0.9, round(SEUIL, 2)}):
        cl = [classe_lame(etendue_mm(toutes_h1[toutes_h1.slide == x], "p", s)) for x in l1.slide]
        print(f"seuil {s:.2f} : {np.mean(np.array(cl) == l1.classe_officielle.values):.0%} des lames de l'hôpital 1 bien classées")
    l1["mm_prédit"] = [etendue_mm(toutes_h1[toutes_h1.slide == x], "p", SEUIL_LAME) for x in l1.slide]
    l1["classe_prédite"] = l1.mm_prédit.map(classe_lame)
    display(l1[["slide", "nom", "classe_officielle", "mm_vrai", "classe_grille", "mm_prédit", "classe_prédite"]].round(2))
    display(confusion_lames(l1.classe_officielle, l1.classe_prédite))

### Stade pN

On vérifie d'abord que les règles redonnent le stade officiel des 100 patientes à partir des classes officielles. Puis on remplace, pour les lames annotées, la classe officielle par une classe estimée, et on compte les stades pN qui changent. Une **sous-stadification** (stade estimé plus bas que le vrai) est l'erreur la plus grave : la patiente risque un traitement insuffisant.

In [ ]:
# Règles pN simplifiées de CAMELYON17 (5 ganglions par patiente)
def stade_pN(classes):
    classes = list(classes)
    macro, micro, itc = classes.count("macro"), classes.count("micro"), classes.count("itc")
    atteints = macro + micro                     # les cellules isolées ne comptent pas
    if atteints == 0:
        return "pN0(i+)" if itc else "pN0"
    if macro == 0:
        return "pN1mi"
    return "pN1" if atteints <= 3 else "pN2"

gl = stages[stages.patient.str.endswith(".tif")].copy(); gl["pid"] = gl.patient.str[:11]
pt = stages[stages.patient.str.endswith(".zip")].copy(); pt["pid"] = pt.patient.str[:11]
recalcule = gl.groupby("pid").stage.apply(stade_pN)
CONCORDANCE_PN = (recalcule == pt.set_index("pid").stage).mean()
print("Concordance des règles avec le stade officiel :", CONCORDANCE_PN)

ORDRE_PN = ["pN0", "pN0(i+)", "pN1mi", "pN1", "pN2"]
def effet_pN(nouvelles):
    """nouvelles : {nom de lame : classe estimée}. Recalcule le stade pN des patientes concernées et liste les changements."""
    g = gl.assign(stage=gl.patient.map(nouvelles).fillna(gl.stage))
    concernees = g[g.patient.isin(list(nouvelles))].pid.unique()
    ch = pd.DataFrame({"officiel": pt.set_index("pid").stage, "recalculé": g.groupby("pid").stage.apply(stade_pN)}).loc[concernees]
    ch = ch[ch.officiel != ch.recalculé]
    ch["erreur"] = np.where(ch.recalculé.map(ORDRE_PN.index) < ch.officiel.map(ORDRE_PN.index),
                            "sous-stadification (traitement insuffisant)", "sur-stadification (traitement en trop)")
    print(f"{len(ch)} stades pN changés sur {len(concernees)} patientes concernées")
    return ch

print("\nClasses issues des labels vrais (plafond de la méthode, 50 lames) :")
PN_GRILLE = effet_pN(dict(zip(lames.nom, lames.classe_grille))); display(PN_GRILLE)
if LAMES_H1:
    print("\nClasses prédites par le modèle (10 lames de l'hôpital 1) :")
    PN_H1 = effet_pN(dict(zip(l1.nom, l1.classe_prédite))); display(PN_H1)

## 9. Synthèse pour le follow-up

Rassemble les chiffres demandés au follow-up dans `resultats_followup.md` et les figures dans `figures/`. Sur Colab, une archive `followup.zip` est téléchargée automatiquement.

In [ ]:
from IPython.display import Markdown

def table_md(d):
    """DataFrame → tableau Markdown, sans dépendance supplémentaire."""
    d = d.reset_index()
    fmt = lambda v: "" if pd.isna(v) else f"{v:.3f}" if isinstance(v, float) else str(v)
    return "\n".join(["| " + " | ".join(map(str, d.columns)) + " |", "|" + "---|" * len(d.columns)]
                     + ["| " + " | ".join(fmt(v) for v in r) + " |" for r in d.itertuples(index=False)])

decoupage = df.groupby("ens").agg(patchs=("label", "size"), lames=("slide", "nunique"),
                                  patientes=("patient", "nunique"), part_tumorale=("label", "mean")).round(2)
comp = pd.DataFrame(RESULTATS).pivot_table(index="modèle", columns=["ensemble", "hôpital"], values="ROC-AUC").round(3)
comp.columns = [f"{e} · {h}" for e, h in comp.columns]
auc_ref = {s: roc_auc_score(y[s], ref.predict_proba(X[s])[:, 1]) for s in ["val_lames", "val_ood"]}
auc_lames = roc_auc_score(df[df.ens == "val_lames"].label, P[CHOIX]["val_lames"])

texte = [
    "# Résultats pour le follow-up",
    f"Généré le {time.strftime('%d/%m/%Y à %H:%M')} — QUICK={QUICK}, époques {EPOCHS_HEAD}+{EPOCHS_FT}, TAILLE={TAILLE}, "
    f"graine {SEED}, calcul sur {DEVICE}.",
    "## 1. Découpage par patiente", table_md(decoupage),
    "## 2. Trois constats d'exploration", "\n".join(f"{k}. {c}" for k, c in enumerate(CONSTATS, 1)),
    "## 3. Référence simple (histogrammes de couleur + régression logistique)",
    f"ROC-AUC : val_lames {auc_ref['val_lames']:.3f} · val_ood (hôpital 1) {auc_ref['val_ood']:.3f}",
    "## 4. CNN par transfert (ResNet18) : ROC-AUC par hôpital", table_md(comp),
    f"- Modèle retenu (meilleure ROC-AUC sur val_ood) : **{CHOIX}** — val_lames {auc_lames:.3f}, val_ood {auc_ood[CHOIX]:.3f} "
    f"(écart val_lames − val_ood : {auc_lames - auc_ood[CHOIX]:+.3f})",
    f"- Seuil pour un rappel ≥ {CIBLE_RAPPEL:.0%} sur val_ood : **{SEUIL:.2f}** → rappel {sens:.3f}, spécificité {spec:.3f}",
    f"- Matrice de confusion val_ood (VN, FP, FN, VP) : {vn}, {fp}, {fn}, {vp}",
    f"- Précision attendue si 1 % des patchs étaient tumoraux : {ppv[prev == .01][0]:.1%}",
    "## 5. Patch → lame → patiente",
    f"- Plafond de la méthode (labels vrais, 50 lames) : {ACCORD_GRILLE:.0%} des lames bien classées ; "
    f"{len(PN_GRILLE)} stades pN changés, dont {PN_GRILLE.erreur.str.startswith('sous').sum()} sous-stadifications",
    f"- Règles pN : {CONCORDANCE_PN:.0%} de concordance avec le stade officiel des 100 patientes",
]
if LAMES_H1:
    texte += [f"- Modèle, lames de l'hôpital 1 (seuil {SEUIL_LAME}) : {(l1.classe_prédite == l1.classe_officielle).sum()} / {len(l1)} "
              f"bien classées ; {len(PN_H1)} stades pN changés, dont {PN_H1.erreur.str.startswith('sous').sum()} sous-stadifications",
              table_md(confusion_lames(l1.classe_officielle, l1.classe_prédite))]
texte += ["## Figures", "\n".join(f"- figures/{f}" for f in sorted(os.listdir("figures")))]
texte = "\n\n".join(texte)
open("resultats_followup.md", "w").write(texte)
with zipfile.ZipFile("followup.zip", "w") as z:
    z.write("resultats_followup.md")
    for f in sorted(os.listdir("figures")):
        z.write(f"figures/{f}")
try:
    from google.colab import files
    files.download("followup.zip")
except ImportError:
    print("Hors Colab : résultats dans resultats_followup.md et figures/")
display(Markdown(texte))

## 10. Test final : hôpital 2, jamais vu

À n'exécuter **qu'une fois**, quand le modèle (`CHOIX`), le seuil des patchs (`SEUIL`) et le seuil des lames (`SEUIL_LAME`) sont fixés sur `val_ood`. Le test télécharge aussi tous les patchs des 10 lames de l'hôpital 2 (~1,9 Go).

In [ ]:
LANCER_TEST = False
if LANCER_TEST:
    d_t = df[df.ens == "test"]; p_t = probas(modele, chargeur("test", tf_eval))
    p_ref = ref.predict_proba(np.stack(d_t.image.map(hist_couleur).values))[:, 1]
    print("Patchs, modèle retenu :"); display(evaluer(d_t, p_t, SEUIL))
    print(confusion_matrix(d_t.label, p_t >= SEUIL, labels=[0, 1]))
    print("Patchs, référence :"); display(evaluer(d_t, p_ref))

    toutes_h2 = predire_lames_hf("test", 2, modele, tf_eval, f"predictions_{CHOIX}_H2.csv")
    l2 = lames[lames.slide.isin(toutes_h2.slide.unique())].copy()
    l2["mm_prédit"] = [etendue_mm(toutes_h2[toutes_h2.slide == x], "p", SEUIL_LAME) for x in l2.slide]
    l2["classe_prédite"] = l2.mm_prédit.map(classe_lame)
    display(l2[["slide", "nom", "classe_officielle", "mm_vrai", "mm_prédit", "classe_prédite"]].round(2))
    display(confusion_lames(l2.classe_officielle, l2.classe_prédite))
    display(effet_pN(dict(zip(l2.nom, l2.classe_prédite))))

## 11. Et maintenant

- Répéter l'entraînement avec plusieurs graines (`SEED`) : l'écart entre variantes est-il plus grand que l'écart entre graines ?
- Essayer une vraie normalisation de la coloration (Macenko, Reinhard), une entrée 224 × 224 ou une autre architecture, et mesurer l'effet **par hôpital**.
- Améliorer l'estimation de l'étendue : lissage des probabilités, taille minimale de zone, seuil de lame mieux justifié.
- Discuter : faux négatifs, rôle du pathologiste, jeu équilibré contre réalité clinique, lames négatives absentes, cadre réglementaire (dispositif médical, AI Act).

Sources : CAMELYON17 (CC0), https://camelyon17.grand-challenge.org ; Camelyon17-WILDS, Koh et al., ICML 2021 (annexe E.2 pour la géométrie des patchs) ; miroir Hugging Face `wltjr1007/Camelyon17-WILDS`.